# Module 06 Lab - Regression and Classification Models**Objective:** To understand the difference between regression and classification and to build your first linear models for both tasks.**In this lab, you will write the code to train and evaluate the models.**

## Part 1: Regression vs. ClassificationThis is the most fundamental distinction between types of supervised learning problems.*   **Regression:** The goal is to predict a **continuous numerical value**.     *   *Examples:* Predicting the price of a house, the temperature tomorrow, or the stock price.*   **Classification:** The goal is to predict a **discrete category or class label**.    *   *Examples:* Predicting if an email is spam or not spam, if a flower is a setosa, versicolor, or virginica, or if a customer will churn or not.**In this lab, we will tackle one of each.**

## Part 2: Linear Regression**Concept:** Linear Regression is used to predict a continuous value. It works by finding the best-fitting straight line through the data points. The model learns a "slope" (coefficient) for each feature and an "intercept".*   **Problem:** We will predict the `Fare` of a Titanic passenger based on their `Age` and `Pclass`.

In [ ]:
import numpy as np  # added: needed for np.sqrt() in Task 1
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Load and prepare the data
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# For simplicity, we'll drop rows with missing age
df.dropna(subset=['Age'], inplace=True)

# Define features and target
features = ['Age', 'Pclass']
target_reg = 'Fare'

X_reg = df[features]
y_reg = df[target_reg]

# Split the data (80% train, 20% test; random_state makes the split reproducible)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

### Task 1: Train and Evaluate a Linear Regression Model**Your Task:**1.  Create an instance of the `LinearRegression` model.2.  Train the model using the training data (`X_train_reg`, `y_train_reg`).3.  Make predictions on the test data.4.  Evaluate the model using `mean_squared_error`. This metric tells us the average of the squared differences between the predicted and actual values.

In [ ]:
# --- Task 1: Linear Regression (Great) ---

# 1. Create the model instance
lr_model = LinearRegression()

# 2. Train the model on the training data
lr_model.fit(X_train_reg, y_train_reg)

# 3. Make predictions on the test data
y_pred_reg = lr_model.predict(X_test_reg)

# 4. Evaluate the model
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
print(f"Mean Squared Error for Fare Prediction: {mse:.2f}")
print(f"The square root of this (RMSE) is {rmse:.2f}, meaning our model is off by about ${rmse:.0f} on average.")

# --- Knowledge Check Q2: model coefficients ---
print("Coefficients:", lr_model.coef_)
for name, coef in zip(features, lr_model.coef_):
    print(f"  {name}: {coef:.2f}")
print(f"Intercept: {lr_model.intercept_:.2f}")

## Part 3: Logistic Regression**Concept:** Despite its name, Logistic Regression is a **classification** algorithm. It works by calculating the probability that a given input belongs to a certain class. It's one of the most widely used and interpretable classification models.*   **Problem:** We will predict whether a passenger `Survived` based on their `Age`, `Pclass`, and `Sex`.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Prepare data for classification
# Encode 'Sex' as numbers. Mapping only if still text, so re-running this cell
# doesn't turn the column into NaN (map() on 0/1 values would return NaN).
if df['Sex'].dtype == object:
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# Define features and target
features_cls = ['Age', 'Pclass', 'Sex']
target_cls = 'Survived'

X_cls = df[features_cls]
y_cls = df[target_cls]

# Split the data
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

### Task 2: Train and Evaluate a Logistic Regression Model**Your Task:**1.  Create an instance of the `LogisticRegression` model.2.  Train the model using the classification training data.3.  Make predictions on the test data.4.  Evaluate the model using `accuracy_score`.

In [ ]:
# --- Task 2: Logistic Regression (Karima) ---

# 1. Create the model instance (max_iter raised so the solver converges)
log_model = LogisticRegression(max_iter=1000)

# 2. Train the model
log_model.fit(X_train_cls, y_train_cls)

# 3. Make predictions
y_pred_cls = log_model.predict(X_test_cls)

# 4. Evaluate the model
accuracy = accuracy_score(y_test_cls, y_pred_cls)
print(f"Accuracy for Survival Prediction: {accuracy:.2%}")

## 📝 Knowledge Check

**1. In your own words, what is the key difference between a regression problem and a classification problem?**

A regression problem predicts a continuous number that can take any value in a range, like a passenger's fare. A classification problem predicts which category something belongs to, like whether a passenger survived (1) or did not survive (0). The key difference is the type of answer: regression asks "how much?" and classification asks "which one?"

**2. What do the numbers in `lr_model.coef_` represent?**

`lr_model.coef_` holds one number per feature, in the same order as `features = ['Age', 'Pclass']`. The first value is the coefficient for `Age` and the second is for `Pclass`. Each one tells us how much the predicted fare changes when that feature goes up by 1 while the other feature stays the same:

* **Age:** the change in predicted fare for each additional year of age, holding passenger class constant.
* **Pclass:** the change in predicted fare when moving up one class number (for example, from 1st class to 2nd class), holding age constant. This coefficient is negative and much larger in size, which makes sense: higher class numbers mean cheaper tickets, so passenger class has a much stronger effect on fare than age does.

**3. Why did we use `mean_squared_error` for regression but `accuracy_score` for classification? Why wouldn't accuracy be a good metric for fare prediction?**

Regression predicts a continuous number, like a passenger's fare, so a prediction is almost never exactly right. What matters is how far off it is. MSE averages the squared differences between predicted and actual values, so it measures the size of the errors and penalizes big mistakes more heavily. Classification predicts a category (survived = 1, did not survive = 0), so each prediction is simply right or wrong, and accuracy (the share of correct predictions) is a natural measure.

Accuracy only counts exact matches. A predicted fare of \$30.10 when the real fare is \$30.00 would count as wrong, the same as a prediction of \$300. Because continuous predictions almost never match exactly, accuracy would be near 0% even for a good model, and it would not tell us whether the model is close or far off. MSE captures that difference.